# `the_one.ipynb` — optimized four-factor socle multiplicities

This notebook optimizes the **specific** coefficient implemented in the original `the_one.ipynb`, **not** the three-factor Hoyt formula. It requires a **SageMath kernel**, because it uses `sage.libs.lrcalc.lrcalc`.

For outer partitions $\lambda,\mu$ and residual partitions $\lambda',\mu'$, the original code computes

$$
T_k^{\lambda,\mu}(\lambda',\mu')=
\sum_{\substack{\delta,\gamma,\,p,q\ge0\\
|\delta|+2|\gamma|+p+q=k}}
N^{\lambda}_{\lambda',\gamma,\delta,(1^p)}
N^{\mu}_{\mu',\gamma,\delta,(1^q)}.
$$

Here $(1^p)$ is a **column of $p$ boxes**, with $(1^0)=\varnothing$. The factors $\delta,\gamma$ are shared, but $p,q$ can differ. This is **not** the old Hoyt sum, and in particular **$k$ may exceed $\min(|\lambda|,|\mu|)$**.

## Algebraic optimization

Let $A=|\lambda|-|\lambda'|$ and $B=|\mu|-|\mu'|$. The degree constraints imply

$$
A=|\delta|+|\gamma|+p,\qquad
B=|\delta|+|\gamma|+q,\qquad
k=|\delta|+2|\gamma|+p+q.
$$

Writing $d=|\delta|$ and $g=|\gamma|$, eliminate $p,q$:

$$
\boxed{d=A+B-k},\qquad
p=A-d-g,\qquad q=B-d-g.
$$

Thus `solveOne()` needs only a **single loop over $g$**, rather than nested degree searches.

To avoid evaluating a four-factor LR coefficient repeatedly, let $\xi\prec_p\lambda$ mean $\lambda/\xi$ is a vertical $p$-strip. Pieri's rule and associativity give

$$
N^\lambda_{\lambda',\gamma,\delta,(1^p)}
=\sum_{\nu\vdash d+g}c^\nu_{\delta,\gamma}
\underbrace{\sum_{\substack{\xi\subseteq\lambda\\\lambda/\xi\ \mathrm{vertical}\ p\mathrm{-strip}}}
 c^\xi_{\nu,\lambda'}}_{L_{\lambda,p}(\lambda',\nu)}.
$$

Define $R_{\mu,q}(\mu',\omega)$ similarly and a **split-degree kernel**

$$
\boxed{K_{d,g}(\nu,\omega)
=\sum_{\substack{\delta\vdash d\\\gamma\vdash g}}
 c^\nu_{\delta,\gamma}c^\omega_{\delta,\gamma}}.
$$

The optimized coefficient is the exact contraction

$$
\boxed{
T_k^{\lambda,\mu}(\lambda',\mu')
=\sum_{\substack{d,g,p,q\ge0\\d+2g+p+q=k}}
\sum_{\nu,\omega\vdash d+g}
L_{\lambda,p}(\lambda',\nu)\,K_{d,g}(\nu,\omega)\,
R_{\mu,q}(\mu',\omega).
}
$$

Unlike the Hoyt kernel $K_n$, **we must preserve the split $(d,g)$**, since $\gamma$ carries weight two in the layer index.


In [1]:
from collections import defaultdict
from functools import lru_cache
from pathlib import Path
import sage.libs.lrcalc.lrcalc as lrcalc

def part(p):
    p = tuple(int(x) for x in p)
    if any(x <= 0 for x in p) or any(a < b for a, b in zip(p, p[1:])):
        raise ValueError(f"Invalid partition: {p}")
    return p

def degree(p):
    return sum(p)

@lru_cache(maxsize=None)
def partitions_of(n, cap=None):
    n = int(n)
    if n < 0:
        return ()
    if n == 0:
        return ((),)
    top = n if cap is None else min(n, int(cap))
    return tuple((i,) + rest
                 for i in range(top, 0, -1)
                 for rest in partitions_of(n-i, i))

@lru_cache(maxsize=None)
def coproduct(p):
    """All *ordered* LR coproduct pairs (a,b) -> c^p_{a,b}."""
    p = part(p)
    return {(part(a), part(b)): int(c)
            for (a,b), c in lrcalc.coprod(p, all=1).items()}

@lru_cache(maxsize=None)
def lr(outer, a, b):
    outer, a, b = map(part, (outer, a, b))
    if degree(outer) != degree(a) + degree(b):
        return 0
    return int(lrcalc.lrcoef_unsafe(outer, a, b))

# Compatibility with the original, but no brute-force intermediate partition loops.
@lru_cache(maxsize=None)
def lrcoef4(a, b, c, d, outer):
    a, b, c, d, outer = map(part, (a, b, c, d, outer))
    if degree(a)+degree(b)+degree(c)+degree(d) != degree(outer):
        return 0
    # Simpler, robust fallback using only actual product support.
    prod1 = lrcalc.mult(a, b)
    prod2 = lrcalc.mult(c, d)
    return sum(int(v1)*int(v2)*lr(outer, part(nu), part(om))
               for nu, v1 in prod1.items()
               for om, v2 in prod2.items())

print("SageMath four-factor LR helpers ready")

SageMath four-factor LR helpers ready


## Closed-form degree solutions and sparse Pieri/coproduct vectors

In [2]:
def solveOne(k, k1, k2, k3, k4):
    """Exact degree solutions (same dictionary fields as original the_one.ipynb)."""
    k, k1, k2, k3, k4 = map(int, (k, k1, k2, k3, k4))
    A, B = k1-k2, k3-k4
    d = A+B-k
    if min(k, A, B) < 0 or d < 0 or d > min(A,B):
        return []
    result = []
    for g in range(min(A-d, B-d)+1):
        p, q = A-d-g, B-d-g
        assert d + 2*g + p + q == k
        result.append({"deg δ": d, "deg γ": g, "p": p, "q": q,
                       "deg λ'": k2, "deg μ'": k4})
    return result

@lru_cache(maxsize=None)
def vertical_predecessors(outer, p):
    """All xi such that outer/xi is a vertical p-strip (one box per row)."""
    outer, p = part(outer), int(p)
    if p < 0 or p > len(outer):
        return ()
    ans = []
    def visit(i, previous, todo, values):
        if todo < 0 or todo > len(outer)-i:
            return
        if i == len(outer):
            if todo == 0:
                ans.append(tuple(x for x in values if x))
            return
        row = outer[i]
        for removed in (0, 1):
            new = row - removed
            if new >= 0 and new <= previous:
                visit(i+1, new, todo-removed, values+(new,))
    visit(0, float('inf'), p, ())
    return tuple(ans)

@lru_cache(maxsize=None)
def coproduct_by_left_degree(outer, n):
    """Map right factor -> {left factor: coefficient}, |left|=n."""
    outer, n = part(outer), int(n)
    grouped = defaultdict(dict)
    for (nu, lp), coeff in coproduct(outer).items():
        if degree(nu) == n:
            grouped[lp][nu] = int(coeff)
    return dict(grouped)

@lru_cache(maxsize=None)
def side_vectors(outer, p, n):
    """Map residual partition -> {nu : sum_{xi vertical p} c^xi_{nu,residual}}."""
    outer, p, n = part(outer), int(p), int(n)
    if p < 0 or n < 0 or p+n > degree(outer):
        return {}
    grouped = defaultdict(lambda: defaultdict(int))
    for xi in vertical_predecessors(outer,p):
        for lp, vec in coproduct_by_left_degree(xi,n).items():
            for nu, mult in vec.items():
                grouped[lp][nu] += mult
    return {lp: dict(vec) for lp, vec in grouped.items()}

@lru_cache(maxsize=None)
def split_coproduct(outer, d):
    """Restrict ordered pairs (delta,gamma) to |delta|=d."""
    outer, d = part(outer), int(d)
    return {pair: int(c) for pair,c in coproduct(outer).items()
            if degree(pair[0]) == d}

@lru_cache(maxsize=None)
def split_kernel(d, g, nu, omega):
    """K_{d,g}(nu,omega). d and g cannot be summed together."""
    d, g, nu, omega = int(d), int(g), part(nu), part(omega)
    if degree(nu) != d+g or degree(omega) != d+g:
        return 0
    if d == 0 or g == 0:
        return int(nu == omega)
    # Swapping d and g gives the same kernel, by commutativity of Schur products.
    d = min(d,g)
    if omega < nu:
        nu,omega = omega,nu
    a, b = split_coproduct(nu, d), split_coproduct(omega, d)
    if len(a) > len(b):
        a,b=b,a
    return sum(c*b.get(pair,0) for pair,c in a.items())

print("Closed-form degrees, vertical-strip vectors, and split kernels ready")

Closed-form degrees, vertical-strip vectors, and split kernels ready


## Complete optimized layer calculation

In [3]:
@lru_cache(maxsize=None)
def row_data(k, lam, mu):
    """Sparse {(lambda_prime, mu_prime): multiplicity}, exactly the_one's T_k."""
    k, lam, mu = int(k), part(lam), part(mu)
    nl, nm = degree(lam), degree(mu)
    if k < 0 or k > nl+nm:
        return {}
    result = defaultdict(int)
    # k = d + 2g + p + q. n=d+g is the shared Schur-product degree.
    for d in range(min(k,nl,nm)+1):
        for g in range((k-d)//2+1):
            n = d+g
            if n > min(nl,nm):
                continue
            strip_total = k-d-2*g  # p + q
            p_min = max(0, strip_total-min(len(mu),nm-n))
            p_max = min(strip_total, len(lam),nl-n)
            for p in range(p_min,p_max+1):
                q = strip_total-p
                left = side_vectors(lam,p,n)
                right = side_vectors(mu,q,n)
                if not left or not right:
                    continue
                nus = {nu for vec in left.values() for nu in vec}
                oms = {om for vec in right.values() for om in vec}
                nonzero_K = {(nu,om): split_kernel(d,g,nu,om)
                             for nu in nus for om in oms}
                for lp, lvec in left.items():
                    for mp, rvec in right.items():
                        total = sum(ca*cb*nonzero_K[(nu,om)]
                                    for nu,ca in lvec.items()
                                    for om,cb in rvec.items())
                        if total:
                            result[(lp,mp)] += total
    return dict(result)

def CalcSoc(k, lam, lamP, mu, muP):
    """Original function signature, now a lookup in a cached complete row."""
    return int(row_data(k,part(lam),part(mu)).get((part(lamP),part(muP)),0))

def max_degree(lam,mu):
    """Highest nonzero layer; note upper bound |lam|+|mu|, not min degrees."""
    lam,mu=part(lam),part(mu)
    return next((k for k in range(degree(lam)+degree(mu),-1,-1)
                 if row_data(k,lam,mu)), 0)

def degree_rows(lam,mu,include_zeros=False):
    lam,mu=part(lam),part(mu)
    for k in range(degree(lam)+degree(mu)+1):
        data=row_data(k,lam,mu)
        if data or include_zeros:
            yield k,data

print("Exact sparse multiplicity evaluator ready")

Exact sparse multiplicity evaluator ready


## Table generation

The original `MakeTable_tex` stops at its first empty row. This notebook instead checks **every possible** $k\in[0,|\lambda|+|\mu|]$, so it never assumes layers are consecutive. `write_table` creates parent directories and defaults to overwriting files rather than accidentally appending stale results.


In [4]:
def latex_part(p):
    p=part(p)
    return r"\varnothing" if not p else "("+",".join(map(str,p))+")"

def direct_sum(k,lam,mu,zero="0"):
    entries=row_data(k,part(lam),part(mu))
    terms=[]
    for (lp,mp),mult in sorted(entries.items(),reverse=True):
        prefix="" if mult==1 else str(mult)+r"\,"
        terms.append(prefix+r"V^{"+latex_part(lp)+","+latex_part(mp)+"}")
    return r" \oplus ".join(terms) if terms else zero

def DirectSumMethod(k,lam,mu):
    """Original interface: return empty string for an empty layer."""
    return direct_sum(k,lam,mu,zero="")

def table_tex(lam,mu,include_zeros=False):
    lam,mu=part(lam),part(mu)
    rows=[]
    for k in range(degree(lam)+degree(mu),-1,-1):
        expression=direct_sum(k,lam,mu)
        if include_zeros or expression!="0":
            rows.append(rf"$k={k}:\quad {expression}$ \\ \hline")
    title=rf"Table for $\lambda={latex_part(lam)}$, $\mu={latex_part(mu)}$"
    return "\n".join([
        r"\begin{table}[htbp]", r"\centering",
        rf"\caption{{{title}}}",
        r"\renewcommand{\arraystretch}{1.35}",
        r"\begin{tabular}{|>{\centering\arraybackslash}p{0.93\linewidth}|}",
        r"\hline", *rows, r"\end{tabular}", r"\end{table}", ""
    ])

def write_table(lam,mu,filename,append=False,include_zeros=False):
    path=Path(filename)
    path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('a' if append else 'w',encoding='utf8') as f:
        f.write(table_tex(lam,mu,include_zeros=include_zeros))
        f.write("\n")
    return path

def MakeTable_tex(lam,mu,filename):
    """Compatibility wrapper: old method appended by default."""
    return write_table(lam,mu,filename,append=True)

def MakeTable(k,lam,mu):
    """Print the LaTeX table for one pair (k retained for API compatibility)."""
    print(table_tex(lam,mu))

def Master1(k,lam,lamP,mu,muP,ex_string=''):
    """Original convenience entry point."""
    print(CalcSoc(k,lam,lamP,mu,muP))

def ones_partition(x):
    x=int(x)
    if x<0: raise ValueError('x must be nonnegative')
    return (1,)*x

def generate_tables(max_lam_degree,max_mu_degree,filename,
                    include_zeros=False, exploit_symmetry=False):
    from itertools import product
    max_lam_degree,max_mu_degree=map(int,(max_lam_degree,max_mu_degree))
    path=Path(filename)
    path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('w',encoding='utf8') as f:
        f.write("% Requires the LaTeX array package\n")
        for n in range(max_lam_degree+1):
            for m in range(max_mu_degree+1):
                if exploit_symmetry and n>m:
                    continue
                for lam in partitions_of(n):
                    for mu in partitions_of(m):
                        if exploit_symmetry and n==m and lam>mu:
                            continue
                        f.write(table_tex(lam,mu,include_zeros=include_zeros))
                        f.write("\n%--------------------\n")
    return path

def generateTables(n,m,file="tex_tables/t_all.tex"):
    return generate_tables(n,m,file)

def le_generateTables(max_m,max_n,filename="tex_tables/t_all.tex"):
    return generate_tables(max_n,max_m,filename)

# Examples, uncomment to execute:
# print(solveOne(2, 1, 0, 1, 0))
# print(row_data(2, (1,), (1,)))
# print(table_tex((1,1), (2,)))
# print(write_table((1,1), (2,), "tex_tables/example.tex"))
# print(generate_tables(3,3,"tex_tables/t_all.tex", exploit_symmetry=True))

## Compact LaTeX export: only $k=0$ and $k=k_{\max}$

This *additional* exporter leaves `table_tex`, `write_table`, and `generate_tables` unchanged.
It prints exactly two **logical layers** (one if $k_{\max}=0$), omitting all intermediate
layer terms. Long $\oplus$ sums are split over `longtable` continuation rows, which
may break across pages instead of overflowing a large table cell.

By default, `.tex` exports are complete compilable LaTeX documents using
`amsmath`, `booktabs` and `longtable`.

```python
# Preview one compact table fragment:
print(table_top_bottom_tex((2,1,1), (3,1,1)))

# Generate one complete .tex document:
write_top_bottom_table((2,1,1), (3,1,1),
                       'tex_tables/top_bottom_example.tex')

# Batch export all pairs through degree 9 (can take time):
generate_top_bottom_tables(9, 9, 'tex_tables/top_bottom_deg9.tex',
                           exploit_symmetry=True)

# More line wrapping for large top-layer decompositions:
# table_top_bottom_tex(lam, mu, max_terms_per_line=1, max_chars=48)
```


In [5]:
# Only the two extremal nonzero layers: k_max and k=0.
# Requires the existing functions part, degree, latex_part, row_data,
# max_degree, and partitions_of from the optimized notebook.
from pathlib import Path


def _extremal_terms(k, lam, mu):
    """LaTeX terms for the actual sparse layer, without recomputation."""
    data = row_data(int(k), part(lam), part(mu))
    terms = []
    for (lp, mp), multiplicity in sorted(data.items(), reverse=True):
        factor = '' if int(multiplicity) == 1 else str(int(multiplicity)) + r'\,'
        terms.append(factor + r'V^{' + latex_part(lp) + ',' + latex_part(mp) + '}')
    return terms or ['0']


def _wrap_extremal_terms(terms, max_terms_per_line=2, max_chars=68):
    """Split long ⊕ sums at term boundaries; do not truncate any terms."""
    if max_terms_per_line < 1 or max_chars < 1:
        raise ValueError('max_terms_per_line and max_chars must be positive')
    chunks, current = [], []
    for term in terms:
        candidate = current + [term]
        text_length = len(r' \oplus '.join(candidate))
        if current and (len(candidate) > max_terms_per_line or text_length > max_chars):
            chunks.append(current)
            current = [term]
        else:
            current = candidate
    if current:
        chunks.append(current)
    return chunks


def table_top_bottom_tex(lam, mu, max_terms_per_line=2, max_chars=68):
    """LaTeX longtable with only k=k_max and k=0 (no duplicate if k_max=0).

    Long direct sums are split over continuation lines. A longtable can
    break between these lines at page boundaries.
    """
    lam, mu = part(lam), part(mu)
    top = int(max_degree(lam, mu))
    selected_layers = [top] if top == 0 else [top, 0]
    title = (r'Extremal layers for $\lambda=' + latex_part(lam)
             + r'$, $\mu=' + latex_part(mu) + r'$')
    result = [
        r'\begingroup', r'\small',
        r'\setlength{\LTpre}{6pt}', r'\setlength{\LTpost}{6pt}',
        r'\begin{longtable}{@{}p{0.14\linewidth}@{\hspace{0.8em}}p{0.79\linewidth}@{}}',
        r'\caption{' + title + r'}\\',
        r'\toprule', r'Layer & Decomposition \\', r'\midrule',
        r'\endfirsthead',
        r'\toprule', r'Layer & Decomposition (continued) \\', r'\midrule',
        r'\endhead',
    ]
    for layer_index, k in enumerate(selected_layers):
        chunks = _wrap_extremal_terms(
            _extremal_terms(k, lam, mu),
            max_terms_per_line=max_terms_per_line,
            max_chars=max_chars,
        )
        for i, chunk in enumerate(chunks):
            key = rf'$k={k}$' if i == 0 else ''
            expression = r' \oplus '.join(chunk)
            if i > 0:
                expression = r'{}\oplus ' + expression
            result.append(key + r' & $\displaystyle ' + expression + r'$ \\')
        if layer_index < len(selected_layers) - 1:
            result.append(r'\addlinespace[5pt]\midrule\addlinespace[5pt]')
    result += [r'\bottomrule', r'\end{longtable}', r'\endgroup', '']
    return '\n'.join(result)


def _top_bottom_document_start(title='Extremal socle layers', author=''):
    def escape_latex(s):
        return (str(s).replace('\\', r'\textbackslash{}')
                .replace('&', r'\&').replace('%', r'\%')
                .replace('_', r'\_').replace('#', r'\#'))
    return '\n'.join([
        r'\documentclass[a4paper,11pt]{article}',
        r'\usepackage[margin=22mm]{geometry}',
        r'\usepackage[T1]{fontenc}',
        r'\usepackage{lmodern}',
        r'\usepackage{amsmath,amssymb}',
        r'\usepackage{array,booktabs,longtable}',
        r'\setlength{\LTleft}{0pt}',
        r'\setlength{\LTright}{0pt}',
        r'\title{' + escape_latex(title) + '}',
        r'\author{' + escape_latex(author) + '}',
        r'\date{}',
        r'\begin{document}',
        r'\maketitle',
        r'\noindent Only the initial layer $k=0$ and the highest nonzero '
        r'layer $k=k_{\max}$ are shown.\\[8pt]',
        ''
    ])


def write_top_bottom_table(lam, mu, filename, standalone=True,
                           max_terms_per_line=2, max_chars=68,
                           title=None, author=''):
    """Write one pair's two-layer LaTeX as a compilable file by default."""
    path = Path(filename)
    path.parent.mkdir(parents=True, exist_ok=True)
    body = table_top_bottom_tex(lam, mu, max_terms_per_line, max_chars)
    if standalone:
        text = (_top_bottom_document_start(title or 'Extremal socle layers', author)
                + body + '\n' + r'\end{document}' + '\n')
    else:
        text = body
    path.write_text(text, encoding='utf-8')
    return path


def generate_top_bottom_tables(max_lam_degree, max_mu_degree, filename,
                               exploit_symmetry=False, standalone=True,
                               max_terms_per_line=2, max_chars=68,
                               title=None, author=''):
    """Write every partition-pair table, retaining only k_max and k=0."""
    nmax, mmax = int(max_lam_degree), int(max_mu_degree)
    if nmax < 0 or mmax < 0:
        raise ValueError('Degree bounds must be nonnegative')
    path = Path(filename)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        if standalone:
            f.write(_top_bottom_document_start(
                title or f'Extremal socle layers through degree {max(nmax,mmax)}',
                author))
        for n in range(nmax+1):
            for m in range(mmax+1):
                if exploit_symmetry and n > m:
                    continue
                for lam in partitions_of(n):
                    for mu in partitions_of(m):
                        if exploit_symmetry and n == m and lam > mu:
                            continue
                        f.write(table_top_bottom_tex(lam, mu,
                            max_terms_per_line, max_chars))
                        f.write('\n')
        if standalone:
            f.write(r'\end{document}' + '\n')
    return path


# Optional aliases matching the older make-table naming style.
MakeTable_top_bottom_tex = write_top_bottom_table
generate_tables_top_bottom = generate_top_bottom_tables


## LaTeX output: one pair or all partition pairs

The compact exporter still writes a **standalone LaTeX document** containing
only the $k=k_{\max}$ and $k=0$ layers. Immediately below each table it lists
all top-layer terms and their positive $(\gamma,\delta,p,q)$ realizations.
For each outer pair it prints $\beta=\lambda\cap\mu$ once; a dagger denotes a
factor equal to $\beta$. The dagger legend appears **only once**, at the start
of the complete document. Multiplicity sums are checked internally.

For **one pair**:

```python
print(table_top_bottom_with_factors_tex((3,), (3, 2)))
write_top_bottom_with_factors((3,), (3, 2), 'tex_tables/one_pair.tex')
```

For **every ordered pair** of partitions with $|\lambda|,|\mu|\le9$:

```python
generate_concise_latex_all_pairs(
    max_degree=9,
    filename='tex_tables/all_pairs_degree_le9.tex',
    include_swapped_pairs=True,
    print_latex_source=False,
)
```

The generated `.tex` is a full LaTeX document and can be compiled or imported
into Overleaf. Set `print_latex_source=True` if you explicitly want to print
its entire source into the notebook (large for degree 9). The final notebook
cell runs the degree-9 export automatically.

In [6]:
# === Compact table followed IMMEDIATELY by the factorization of every top summand ===
# Requires the earlier theone_optimized.ipynb cells, including factor_contributions.


def _top_factor_column_tex(p):
    """Display the actual column factor (1^p); use empty partition at p=0."""
    p = int(p)
    if p < 0:
        raise ValueError('A column length cannot be negative')
    return r'\varnothing' if p == 0 else r'(1^{' + str(p) + r'})'


def _top_factor_term_tex(lp, mp, multiplicity):
    prefix = '' if int(multiplicity) == 1 else str(int(multiplicity)) + r'\,'
    return prefix + r'V^{' + latex_part(lp) + ',' + latex_part(mp) + '}'


def top_layer_factor_details_tex(lam, mu, mark_intersection=True, verify=True):
    """Concise top-layer realizations: only gamma, delta, p, q.

    Each positive ordered tuple (delta, gamma, p, q) is shown once for
    each nonzero top-layer term. Verification still checks its weighted
    degree and that all (hidden) LR contributions sum to the multiplicity.
    The dagger legend is printed ONCE by _top_factor_doc_start().
    """
    lam, mu = part(lam), part(mu)
    k = int(max_degree(lam, mu))
    beta = partition_intersection(lam, mu)
    top = row_data(k, lam, mu)
    output = [
        r'\Needspace{7\baselineskip}',
        r'\par\medskip',
        r'\noindent $\beta=\lambda\cap\mu=' + latex_part(beta) + r'$\par',
        r'\medskip',
    ]
    for i, ((lp, mp), multiplicity) in enumerate(sorted(top.items(), reverse=True), 1):
        contributions = factor_contributions(lam, mu, lp, mp, k=k, verify=verify)
        if verify:
            assert contributions, (lam, mu, lp, mp, k)
            assert sum(int(x['contribution']) for x in contributions) == int(multiplicity)
            for x in contributions:
                d, g = x['delta'], x['gamma']
                p, q = int(x['p']), int(x['q'])
                assert degree(d) + 2*degree(g) + p + q == k, x
                assert degree(lp) + degree(d) + degree(g) + p == degree(lam), x
                assert degree(mp) + degree(d) + degree(g) + q == degree(mu), x

        term = _top_factor_term_tex(lp, mp, multiplicity)
        output.extend([
            r'\Needspace{7\baselineskip}',
            r'\noindent\textbf{Term ' + str(i) + r':} $' + term + r'$\par',
            r'\begingroup\footnotesize',
            r'\setlength{\LTpre}{3pt}\setlength{\LTpost}{8pt}',
            r'\renewcommand{\arraystretch}{1.14}',
            r'\begin{longtable}{@{}p{0.34\linewidth}p{0.34\linewidth}cc@{}}',
            r'\toprule',
            r'$\gamma$ & $\delta$ & $p$ & $q$ \\',
            r'\midrule',
            r'\endfirsthead',
            r'\multicolumn{4}{@{}l}{\emph{Term ' + str(i) + r' (continued)}} \\',
            r'\toprule',
            r'$\gamma$ & $\delta$ & $p$ & $q$ \\',
            r'\midrule',
            r'\endhead',
        ])
        for x in contributions:
            gamma = latex_part(x['gamma'])
            delta = latex_part(x['delta'])
            if mark_intersection:
                if x['gamma'] == beta:
                    gamma += r'^{\dagger}'
                if x['delta'] == beta:
                    delta += r'^{\dagger}'
            output.append(
                '$' + gamma + '$ & $' + delta + '$ & '
                + str(int(x['p'])) + ' & ' + str(int(x['q'])) + r' \\'
            )
        output.extend([
            r'\bottomrule',
            r'\end{longtable}',
            # Keep the main extremal tables consecutively numbered.
            r'\addtocounter{table}{-1}',
            r'\endgroup',
            '',
        ])
    return '\n'.join(output)

def table_top_bottom_with_factors_tex(lam, mu, max_terms_per_line=2,
                                      max_chars=68, mark_intersection=True,
                                      verify=True):
    """ONE compact table then its top term-by-term factorizations."""
    compact = table_top_bottom_tex(lam, mu, max_terms_per_line=max_terms_per_line,
                                   max_chars=max_chars)
    detail = top_layer_factor_details_tex(lam, mu,
                                          mark_intersection=mark_intersection,
                                          verify=verify)
    return r'\Needspace{17\baselineskip}' + '\n' + compact + '\n' + detail + '\n\n'


def _top_factor_doc_start(title='Extremal layers and their realizations',
                          author='', mark_intersection=True):
    """One short legend at the beginning of the standalone LaTeX document."""
    s = _top_bottom_document_start(title, author)
    s = s.replace(r'\usepackage{array,booktabs,longtable}',
                  r'\usepackage{array,booktabs,longtable,needspace}')
    legend = (r'\noindent Below each table, every row lists one positive '
              r'top-layer realization $(\gamma,\delta,p,q)$.')
    if mark_intersection:
        legend += (r' A $\dagger$ marks $\gamma$ or $\delta$ equal to '
                   r'$\beta=\lambda\cap\mu$.')
    legend += r'\par\medskip' + '\n'
    marker = r'\maketitle'
    assert marker in s
    return s.replace(marker, marker + '\n' + legend, 1)

def write_top_bottom_with_factors(lam, mu, filename, standalone=True,
                                   max_terms_per_line=2, max_chars=68,
                                   mark_intersection=True, verify=True,
                                   title=None, author=''):
    """Export one complete pair, with table -> term 1 -> term 2 -> ..."""
    path = Path(filename)
    path.parent.mkdir(parents=True, exist_ok=True)
    body = table_top_bottom_with_factors_tex(
        lam, mu, max_terms_per_line=max_terms_per_line,
        max_chars=max_chars, mark_intersection=mark_intersection, verify=verify
    )
    contents = (_top_factor_doc_start(title or 'Extremal socle layers and factors',
                                        author, mark_intersection=mark_intersection)
                + body + r'\end{document}' + '\n') if standalone else body
    path.write_text(contents, encoding='utf-8')
    return path


def generate_top_bottom_with_factors_tables(max_lam_degree, max_mu_degree,
                                             filename, exploit_symmetry=False,
                                             standalone=True,
                                             max_terms_per_line=2, max_chars=68,
                                             mark_intersection=True,
                                             verify=True, title=None, author=''):
    """Export all pairs: TABLE; its term decompositions; NEXT TABLE; etc."""
    nmax, mmax = int(max_lam_degree), int(max_mu_degree)
    if nmax < 0 or mmax < 0:
        raise ValueError('Degree bounds must be nonnegative')
    path = Path(filename)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        if standalone:
            f.write(_top_factor_doc_start(
                title or 'Extremal socle layers and their LR realizations',
                author, mark_intersection=mark_intersection))
        for n in range(nmax+1):
            for m in range(mmax+1):
                if exploit_symmetry and n > m:
                    continue
                for lam in partitions_of(n):
                    for mu in partitions_of(m):
                        if exploit_symmetry and n == m and lam > mu:
                            continue
                        f.write(table_top_bottom_with_factors_tex(
                            lam, mu, max_terms_per_line=max_terms_per_line,
                            max_chars=max_chars,
                            mark_intersection=mark_intersection, verify=verify))
        if standalone:
            f.write(r'\end{document}' + '\n')
    return path

# Discoverable aliases with the earlier naming convention.
generate_tables_top_bottom_with_factors = generate_top_bottom_with_factors_tables
MakeTable_top_bottom_with_factors_tex = table_top_bottom_with_factors_tex


# === Batch export: the same concise tables for ALL partition pairs ===

def generate_concise_latex_all_pairs(max_degree=9,
                                    filename=None,
                                    include_swapped_pairs=True,
                                    print_latex_source=False,
                                    preview_chars=750,
                                    verify=True,
                                    max_terms_per_line=2,
                                    max_chars=68):
    """Write one complete standalone .tex document for |lambda|,|mu| <= N.

    Default at N=9: all 9,409 ORDERED outer pairs, including empty partitions.
    Each pair: extremal table, beta once, each top term's gamma/delta/p/q.

    print_latex_source=True dumps the ENTIRE .tex source to stdout;
    normally leave it False and download/open the generated file instead.
    """
    N = int(max_degree)
    if N < 0:
        raise ValueError('max_degree must be nonnegative')
    if filename is None:
        filename = f'tex_tables/all_pairs_degree_le{N}.tex'
    out = generate_top_bottom_with_factors_tables(
        N, N, filename,
        exploit_symmetry=not include_swapped_pairs,
        standalone=True,
        max_terms_per_line=max_terms_per_line,
        max_chars=max_chars,
        mark_intersection=True,
        verify=verify,
        title=f'Extremal socle layers and top-layer realizations (degree <= {N})',
    )
    print('LaTeX document written:', out.resolve())
    print('Size:', f'{out.stat().st_size:,}', 'bytes')
    if print_latex_source:
        print(out.read_text(encoding='utf-8'))
    elif preview_chars:
        text = out.read_text(encoding='utf-8')
        print('LaTeX source preview (full source in the .tex file):')
        print(text[:int(preview_chars)])
        if len(text) > int(preview_chars):
            print('...')
    return out


def print_concise_latex_for_pair(lam, mu):
    """Print one pair's compact table + all top term realizations as LaTeX."""
    print(table_top_bottom_with_factors_tex(lam, mu))


## Regression against the original defining formula

This checks the **four-factor** coefficient and the original `solveOne` degree enumeration independently of the sparse kernel. Small complete tables are compared entry by entry; the tests do **not** assume the Hoyt `k` bounds.

SageMath is required to execute this cell. The baseline is deliberately slow and should be limited to small degrees.


In [7]:
@lru_cache(maxsize=None)
def reference_four_lr(a,b,c,d,outer):
    a,b,c,d,outer=map(part,(a,b,c,d,outer))
    if degree(a)+degree(b)+degree(c)+degree(d)!=degree(outer):
        return 0
    total=0
    for nu in partitions_of(degree(a)+degree(b)):
        x=lr(nu,a,b)
        if not x: continue
        for om in partitions_of(degree(c)+degree(d)):
            y=lr(om,c,d)
            if y: total+=x*y*lr(outer,nu,om)
    return total

def reference_solveOne(k,k1,k2,k3,k4):
    """Intentionally literal degree loops, independent of the closed-form solver."""
    if k2>k1 or k4>k3 or k<0:
        return []
    out=[]
    for d in range(k+1):
        for g in range((k-d)//2+1):
            p=k1-k2-d-g
            q=k3-k4-d-g
            if min(p,q)<0:continue
            if d+2*g+p+q==k:
                out.append((d,g,p,q))
    return out

def reference_coefficient(k,lam,lp,mu,mp):
    lam,lp,mu,mp=map(part,(lam,lp,mu,mp))
    out=0
    for d,g,p,q in reference_solveOne(k,degree(lam),degree(lp),degree(mu),degree(mp)):
        colp=(1,)*p
        colq=(1,)*q
        for delta in partitions_of(d):
            for gamma in partitions_of(g):
                out += reference_four_lr(lp,gamma,delta,colp,lam)*\
                       reference_four_lr(mp,gamma,delta,colq,mu)
    return out

# Original and optimized solver enumeration must coincide.
for k1 in range(5):
    for k3 in range(5):
        for k2 in range(k1+1):
            for k4 in range(k3+1):
                for k in range(k1+k3+1):
                    got = [(x['deg δ'],x['deg γ'],x['p'],x['q'])
                           for x in solveOne(k,k1,k2,k3,k4)]
                    assert got==reference_solveOne(k,k1,k2,k3,k4), (k,k1,k2,k3,k4)

# Complete multiplicity regression for small source partitions.
for lam in [(),(1,),(2,),(1,1),(2,1)]:
    for mu in [(),(1,),(2,),(1,1)]:
        for k in range(degree(lam)+degree(mu)+1):
            result=row_data(k,lam,mu)
            for lp in (x for n in range(degree(lam)+1) for x in partitions_of(n)):
                for mp in (y for n in range(degree(mu)+1) for y in partitions_of(n)):
                    got=result.get((lp,mp),0)
                    expected=reference_coefficient(k,lam,lp,mu,mp)
                    assert got==expected,(k,lam,lp,mu,mp,got,expected)

assert CalcSoc(2,(1,),(),(1,),())==2
assert max_degree((1,),(1,))>=2
print("Small-degree four-factor regression passed")

Small-degree four-factor regression passed


## Practical notes

- The source `the_one.ipynb` is unchanged. The `solveOne` degree solutions are preserved algebraically.
- Vertical-strip predecessors use Pieri's rule, avoiding brute-force intermediate partitions.
- Sparse LR coproducts and the split kernel are memoized (`functools.lru_cache`).
- `CalcSoc` preserves the original call signature, now reading an entire cached layer instead of recomputing one entry at a time.
- Full layers are examined through $|\lambda|+|\mu|$, not the smaller Hoyt upper bound.
- `lrcalc.coprod(..., all=1)` includes **ordered** pairs; do not insert a factor of two manually.
- Results need validation under **SageMath 10.7**, which is not installed in the current export environment. The mathematical reorganization is exact; a separate pure-Python combinatorial test is also supplied.

When generating large batches, begin with low degrees and inspect `coproduct.cache_info()`, `side_vectors.cache_info()`, `split_kernel.cache_info()`, and `row_data.cache_info()` to understand cache reuse.

## Top-layer intersection test: weak and strong properties

**This is specific to `the_one`, not the Hoyt layer formula.** Its original nonnegative summands are

$$
T_k^{\lambda,\mu}(\lambda',\mu')=
\sum_{\substack{\delta,\gamma,\ p,q\ge 0\\
|\delta|+2|\gamma|+p+q=k}}
N^\lambda_{\lambda',\gamma,\delta,(1^p)}
N^\mu_{\mu',\gamma,\delta,(1^q)}.
$$

Let $\beta=\lambda\cap\mu$ and $k_*=\max\{k:T_k\ne 0\}$. We say that $\beta$ **appears in a top summand** $T_{k_*}^{\lambda,\mu}(\lambda',\mu')$ when at least one **positive** original summand has $\delta=\beta$ or $\gamma=\beta$ (the values of $p,q$ are unrestricted).

- **Weak property:** at least one nonzero top summand contains such a contribution.
- **Strong property:** **every** nonzero top summand contains such a contribution. This does **not** mean every contributing tuple must contain $\beta$.

Unlike Hoyt's unweighted sum, `the_one` has $k=|\delta|+2|\gamma|+p+q$, so we retain the distinction between $\delta$ and $\gamma$, as well as the column factors $(1^p),(1^q)$. `contribution_tuples` reconstructs the original factors from the optimized sparse vectors, and asserts that their multiplicities sum to the value returned by `row_data`.

If $\beta=\varnothing$, a factor can equal it trivially. Set `include_empty_intersection=False` when measuring nontrivial intersection participation.

The display helper `show_top_layer_factors` retrieves all positive terms directly from the original four-factor formula and verifies their sum against each optimized top-layer summand. It prints the factor names **gamma** and **delta** separately because they carry different weights in k.


In [8]:
from itertools import zip_longest
import csv


def partition_intersection(lam, mu):
    """Young-diagram intersection, expressed as an ordinary partition tuple."""
    lam, mu = part(lam), part(mu)
    return tuple(min(a, b) for a, b in zip_longest(lam, mu, fillvalue=0)
                 if min(a, b) > 0)


@lru_cache(maxsize=None)
def side_factor_coefficients(outer, residual, p, d, g):
    r"""Recover N^outer_{residual, gamma, delta, (1^p)}, grouped by (delta,gamma).

    We keep |delta| = d and |gamma| = g separate because gamma has twice
    delta's weight in the_one's layer degree. The original factor order is
    commutative in the Schur ring, but the *names* delta/gamma are not swapped.
    """
    outer, residual = part(outer), part(residual)
    p, d, g = int(p), int(d), int(g)
    output = defaultdict(int)
    for nu, multiplicity in side_vectors(outer, p, d + g).get(residual, {}).items():
        for (delta, gamma), lr_multiplicity in split_coproduct(nu, d).items():
            if degree(gamma) == g:
                output[(delta, gamma)] += int(multiplicity) * int(lr_multiplicity)
    return dict(output)


@lru_cache(maxsize=None)
def contribution_tuples(k, lam, lp, mu, mp):
    r"""Exact nonzero terms: (delta,gamma,p,q,left_N,right_N,product).

    The sum of products equals T_k^{lam,mu}(lp,mp). Each tuple refers to
    one ordered pair (delta,gamma) and one vertical-strip pair (p,q).
    """
    k = int(k)
    lam, lp, mu, mp = map(part, (lam, lp, mu, mp))
    terms = []
    for solution in solveOne(k, degree(lam), degree(lp), degree(mu), degree(mp)):
        d, g, p, q = (int(solution[key]) for key in ('deg δ', 'deg γ', 'p', 'q'))
        left = side_factor_coefficients(lam, lp, p, d, g)
        right = side_factor_coefficients(mu, mp, q, d, g)
        if len(left) > len(right):
            common = (x for x in right if x in left)
        else:
            common = (x for x in left if x in right)
        for delta, gamma in common:
            a, b = int(left[(delta, gamma)]), int(right[(delta, gamma)])
            if a and b:
                terms.append((delta, gamma, p, q, a, b, a * b))
    terms.sort(key=lambda x: (x[0], x[1], x[2], x[3]))
    return tuple(terms)


def factor_contributions(lam, mu, lp, mp, k=None, verify=True):
    """Return the four-factor LR contributions to a selected layer summand."""
    lam, lp, mu, mp = map(part, (lam, lp, mu, mp))
    k = max_degree(lam, mu) if k is None else int(k)
    raw = contribution_tuples(k, lam, lp, mu, mp)
    total = sum(record[-1] for record in raw)
    if verify:
        expected = int(row_data(k, lam, mu).get((lp, mp), 0))
        assert total == expected, (k, lam, lp, mu, mp, total, expected)
    return [dict(delta=delta, gamma=gamma, p=p, q=q,
                 left_coefficient=left, right_coefficient=right,
                 contribution=value)
            for delta, gamma, p, q, left, right, value in raw]


def show_top_layer(lam, mu):
    """Print all top summands and return (top_k, {(lp,mp):multiplicity})."""
    lam, mu = part(lam), part(mu)
    k = max_degree(lam, mu)
    row = row_data(k, lam, mu)
    print('lambda =', lam, 'mu =', mu, 'intersection =', partition_intersection(lam, mu))
    print('Maximum weighted layer k =', k, ' | nonzero top summands =', len(row))
    for (lp, mp), mult in sorted(row.items(), reverse=True):
        print('  (lambda_prime, mu_prime) =', (lp, mp), 'multiplicity =', mult)
    return k, row


def inspect_top_pair(lam, mu, index=0, lp=None, mp=None, show=True):
    """Inspect one top-layer summand, by index or explicitly by lp,mp."""
    lam, mu = part(lam), part(mu)
    k = max_degree(lam, mu)
    items = sorted(row_data(k, lam, mu).items(), reverse=True)
    if lp is None and mp is None:
        if not (0 <= index < len(items)):
            raise IndexError('Top summand index outside range')
        (lp, mp), multiplicity = items[index]
    elif lp is not None and mp is not None:
        lp, mp = part(lp), part(mp)
        multiplicity = int(row_data(k, lam, mu).get((lp, mp), 0))
        if not multiplicity:
            raise ValueError('The selected (lambda_prime,mu_prime) is not in the top layer')
    else:
        raise ValueError('Supply both lp and mp or neither')
    contributions = factor_contributions(lam, mu, lp, mp, k=k)
    selection = dict(lambda_partition=lam, mu_partition=mu,
                     top_k=k, lambda_prime=lp, mu_prime=mp,
                     multiplicity=multiplicity, contributions=contributions)
    if show:
        print('k =', k, '| lambda_prime =', lp, '| mu_prime =', mp,
              '| multiplicity =', multiplicity)
        for item in contributions:
            print('  delta =', item['delta'], 'gamma =', item['gamma'],
                  'p =', item['p'], 'q =', item['q'],
                  '| coefficients =', (item['left_coefficient'],
                                       item['right_coefficient']),
                  '| contribution =', item['contribution'])
    return selection


def beta_membership(selection, beta):
    """Check if beta occurs as delta and/or gamma among positive summand terms."""
    beta = part(beta)
    contributions = selection['contributions']
    as_delta = [r for r in contributions if r['delta'] == beta]
    as_gamma = [r for r in contributions if r['gamma'] == beta]
    witnesses = [r for r in contributions if r['delta'] == beta or r['gamma'] == beta]
    return dict(beta=beta, present_as_delta=bool(as_delta),
                present_as_gamma=bool(as_gamma), present=bool(witnesses),
                witnesses=witnesses, number_of_witnesses=len(witnesses),
                witness_multiplicity=sum(r['contribution'] for r in witnesses))


def test_intersection_properties(lam, mu, show=True):
    """Test both properties for one ordered OUTER partition pair.

    WEAK: at least one nonzero top-layer summand has a positive term with
          delta == beta or gamma == beta.
    STRONG: every nonzero top-layer summand has such a positive term.
    Here beta = lambda intersect mu, and k is the *weighted* the_one index.
    """
    lam, mu = part(lam), part(mu)
    beta = partition_intersection(lam, mu)
    k = max_degree(lam, mu)
    top = row_data(k, lam, mu)
    summands = []
    for (lp, mp), multiplicity in sorted(top.items(), reverse=True):
        raw = contribution_tuples(k, lam, lp, mu, mp)
        total = sum(record[-1] for record in raw)
        assert total == int(multiplicity), (lam, mu, k, lp, mp, total, multiplicity)
        as_delta = any(record[0] == beta for record in raw)
        as_gamma = any(record[1] == beta for record in raw)
        witnesses = tuple(record for record in raw if record[0] == beta or record[1] == beta)
        summands.append(dict(lambda_prime=lp, mu_prime=mp,
                             multiplicity=int(multiplicity),
                             present_as_delta=as_delta, present_as_gamma=as_gamma,
                             present=as_delta or as_gamma,
                             witness_count=len(witnesses),
                             witness_multiplicity=sum(record[-1] for record in witnesses),
                             witness=(witnesses[0] if witnesses else None)))
    # No zero-row pair is expected, but never count an empty top as 'strong'.
    weak = any(record['present'] for record in summands)
    strong = bool(summands) and all(record['present'] for record in summands)
    result = dict(lambda_partition=lam, mu_partition=mu, beta=beta,
                  top_k=k, n_top_summands=len(summands),
                  n_summands_with_beta=sum(r['present'] for r in summands),
                  weak=weak, strong=strong, summands=summands)
    if show:
        print('lambda =', lam, 'mu =', mu, 'beta = lambda ∩ mu =', beta,
              '| top k =', k)
        print('WEAK:', 'PASS' if weak else 'FAIL',
              '| STRONG:', 'PASS' if strong else 'FAIL',
              '| summands with beta =', result['n_summands_with_beta'],
              '/', len(summands))
        for row in summands:
            print('  lambda_prime =', row['lambda_prime'],
                  'mu_prime =', row['mu_prime'],
                  'mult =', row['multiplicity'],
                  '| delta:', 'YES' if row['present_as_delta'] else 'no',
                  '| gamma:', 'YES' if row['present_as_gamma'] else 'no')
    return result


def scan_intersection_properties(max_lambda_degree, max_mu_degree=None,
                                 include_empty_intersection=True,
                                 max_examples=10, csv_prefix=None):
    """Exhaustive ordered-pair test through inclusive degree limits.

    If csv_prefix is supplied, write CSV files for BOTH outer-pair summaries
    and individual top-layer summands. Return full structured results.
    """
    max_lambda_degree = int(max_lambda_degree)
    max_mu_degree = (max_lambda_degree if max_mu_degree is None
                     else int(max_mu_degree))
    if min(max_lambda_degree, max_mu_degree) < 0:
        raise ValueError('Degree bounds must be nonnegative')
    all_lam = [p for n in range(max_lambda_degree + 1) for p in partitions_of(n)]
    all_mu = [p for n in range(max_mu_degree + 1) for p in partitions_of(n)]
    pairs, summands = [], []
    for lam in all_lam:
        for mu in all_mu:
            if not include_empty_intersection and not partition_intersection(lam, mu):
                continue
            result = test_intersection_properties(lam, mu, show=False)
            pairs.append(result)
            for summand in result['summands']:
                summands.append(dict(lambda_partition=lam, mu_partition=mu,
                                     beta=result['beta'], top_k=result['top_k'],
                                     **summand))

    count = len(pairs)
    weak_pass = sum(r['weak'] for r in pairs)
    strong_pass = sum(r['strong'] for r in pairs)
    both_or = sum(s['present'] for s in summands)
    counts = dict(pairs=count, top_summands=len(summands),
                  weak_pass=weak_pass, weak_fail=count-weak_pass,
                  strong_pass=strong_pass, strong_fail=count-strong_pass,
                  summands_with_beta=both_or,
                  weak_rate=(weak_pass/count if count else 0.0),
                  strong_rate=(strong_pass/count if count else 0.0),
                  summand_rate=(both_or/len(summands) if summands else 0.0))
    print('Weighted the_one top layer: k = |delta| + 2|gamma| + p + q')
    print('Ordered pairs:', count,
          '| nonzero top summands:', counts['top_summands'],
          '| include empty beta:', include_empty_intersection)
    print('WEAK:  ', weak_pass, '/', count, '= {:.2f}%'.format(100*counts['weak_rate']))
    print('STRONG:', strong_pass, '/', count, '= {:.2f}%'.format(100*counts['strong_rate']))
    print('Summand-level presence:', both_or, '/', len(summands),
          '= {:.2f}%'.format(100*counts['summand_rate']))
    print('\nExamples failing WEAK:')
    weak_shown = 0
    for r in (r for r in pairs if not r['weak']):
        print('  lambda =', r['lambda_partition'], 'mu =', r['mu_partition'],
              'beta =', r['beta'], 'top k =', r['top_k'])
        weak_shown += 1
        if weak_shown >= max_examples: break
    print('\nExamples passing WEAK but failing STRONG:')
    shown = 0
    for r in (r for r in pairs if r['weak'] and not r['strong']):
        print('  lambda =', r['lambda_partition'], 'mu =', r['mu_partition'],
              'beta =', r['beta'], 'top k =', r['top_k'],
              'witnessed summands =', r['n_summands_with_beta'],
              '/', r['n_top_summands'])
        shown += 1
        if shown >= 10: break
    if csv_prefix is not None:
        prefix = Path(csv_prefix)
        prefix.parent.mkdir(parents=True, exist_ok=True)
        pair_file = prefix.with_name(prefix.name + '_pairs.csv')
        summand_file = prefix.with_name(prefix.name + '_summands.csv')
        pair_keys = ['lambda_partition','mu_partition','beta','top_k',
                     'n_top_summands','n_summands_with_beta','weak','strong']
        summand_keys = ['lambda_partition','mu_partition','beta','top_k',
                        'lambda_prime','mu_prime','multiplicity',
                        'present_as_delta','present_as_gamma','present',
                        'witness_count','witness_multiplicity']
        with pair_file.open('w',newline='',encoding='utf8') as f:
            writer = csv.DictWriter(f, fieldnames=pair_keys)
            writer.writeheader()
            writer.writerows({k: r[k] for k in pair_keys} for r in pairs)
        with summand_file.open('w',newline='',encoding='utf8') as f:
            writer = csv.DictWriter(f, fieldnames=summand_keys)
            writer.writeheader()
            writer.writerows({k: r[k] for k in summand_keys} for r in summands)
        print('CSV files:', pair_file, 'and', summand_file)
    return dict(summary=counts, pairs=pairs, summands=summands)


# --- NEW: Visible, complete factor breakdown of EVERY top-layer summand. ---
def show_top_layer_factors(lam, mu, render="html", verify=True):
    """Display all positive (gamma, delta, p, q) for every top-layer summand.

    Parameters
    ----------
    lam, mu : partitions (tuple or Sage Partition)
        Outer partitions in the weighted four-factor the_one formula.
    render : {'html', 'text', 'none'}
        'html' makes high-contrast Jupyter tables; 'text' prints all values;
        'none' only returns the full structured result (useful for testing).
    verify : bool
        Check every factor's weighted degree, both outer degrees, and that
        all contributions add to the optimized row_data multiplicity.

    Returns
    -------
    dict
        top_k, beta, weak, strong, summands, and all factor records.
        Each summand has a list called 'factors' containing delta, gamma,
        p, q, both generalized LR coefficients, and their product.

    Note
    ----
    A positive factor record corresponds exactly to one nonzero term in
       N^lambda_{lambda',gamma,delta,(1^p)}
       N^mu_{mu',gamma,delta,(1^q)}.
    The factor names delta and gamma MUST NOT be interchanged:
       k = |delta| + 2|gamma| + p + q.
    """
    from html import escape as html_escape

    if render not in ('html', 'text', 'none'):
        raise ValueError("render must be 'html', 'text', or 'none'")
    lam, mu = part(lam), part(mu)
    beta = partition_intersection(lam, mu)
    k = max_degree(lam, mu)
    row = row_data(k, lam, mu)
    results = []
    for (lp, mp), expected in sorted(row.items(), reverse=True):
        factors = factor_contributions(lam, mu, lp, mp, k=k, verify=verify)
        if verify:
            for t in factors:
                d, g, p, q = t['delta'], t['gamma'], t['p'], t['q']
                assert degree(d) + 2 * degree(g) + p + q == k, (lam, mu, lp, mp, t)
                assert degree(lp) + degree(d) + degree(g) + p == degree(lam), t
                assert degree(mp) + degree(d) + degree(g) + q == degree(mu), t
            calculated = sum(t['contribution'] for t in factors)
            assert calculated == int(expected), (lam, mu, lp, mp, calculated, expected)
        with_beta = any(t['delta'] == beta or t['gamma'] == beta for t in factors)
        results.append({
            'lambda_prime': lp, 'mu_prime': mp,
            'multiplicity': int(expected),
            'factors': factors,
            'has_intersection_as_delta': any(t['delta'] == beta for t in factors),
            'has_intersection_as_gamma': any(t['gamma'] == beta for t in factors),
            'has_intersection': with_beta,
        })

    output = {
        'lambda_partition': lam,
        'mu_partition': mu,
        'beta': beta,
        'top_k': k,
        'summands': results,
        'summand_count': len(results),
        'factor_count': sum(len(r['factors']) for r in results),
        'weak': any(r['has_intersection'] for r in results),
        'strong': bool(results) and all(r['has_intersection'] for r in results),
    }

    def part_label(x):
        return '∅' if not x else '(' + ', '.join(map(str, x)) + ')'

    if render == 'text':
        print('TOP LAYER FACTORIZATION FOR ALL SUMMANDS')
        print('lambda =', part_label(lam), '| mu =', part_label(mu),
              '| beta = lambda ∩ mu =', part_label(beta), '| top k =', k)
        print('WEAK:', output['weak'], '| STRONG:', output['strong'])
        for number, s in enumerate(results, 1):
            print('\n[{}] (lambda_prime, mu_prime) = ({}, {}) | multiplicity = {}'.format(
                number, part_label(s['lambda_prime']), part_label(s['mu_prime']), s['multiplicity']))
            print('    beta as delta:', s['has_intersection_as_delta'],
                  '| beta as gamma:', s['has_intersection_as_gamma'])
            for j, t in enumerate(s['factors'], 1):
                print('    {:>2}. gamma = {:<15} delta = {:<15} p = {:<2} q = {:<2} '
                      '| N_lambda = {:<3} N_mu = {:<3} | contribution = {}{}'.format(
                          j, part_label(t['gamma']), part_label(t['delta']),
                          t['p'], t['q'], t['left_coefficient'], t['right_coefficient'],
                          t['contribution'],
                          '   <-- beta' if t['delta'] == beta or t['gamma'] == beta else ''))
            print('    Total =', sum(t['contribution'] for t in s['factors']),
                  '(verified against multiplicity)')

    if render == 'html':
        try:
            from IPython.display import HTML, display
        except ImportError:
            # Running as an ordinary Python script instead of a notebook.
            return show_top_layer_factors(lam, mu, render='text', verify=verify)

        css = ('font-family:Arial,sans-serif;background:#f4f7fa;color:#182b3b;'
               'border:1px solid #ced8e1;border-radius:10px;padding:14px;margin:10px 0;')
        table = ('border-collapse:collapse;width:100%;font-size:13px;'
                 'background:white;color:#172636;')
        cell = ('border:1px solid #d9e2e9;padding:7px 9px;text-align:left;'
                'color:#172636;background:#fff;white-space:nowrap;')
        head = ('border:1px solid #345c79;padding:8px 9px;text-align:left;'
                'background:#174366;color:#fff;white-space:nowrap;')
        section = ['<div style="{}">'.format(css)]
        section.append('<h3 style="margin:0 0 8px;color:#143954;">Top-layer factor breakdown</h3>')
        section.append('<p style="margin:0 0 6px;color:#182b3b;">'
                       '<b>λ</b> = {} &nbsp; <b>μ</b> = {} &nbsp; '
                       '<b>β = λ ∩ μ</b> = {} &nbsp; <b>top k</b> = {}'
                       '</p>'.format(part_label(lam),part_label(mu),part_label(beta),k))
        section.append('<p style="margin:0 0 12px;color:#345168;">'
                       'WEAK: <b>{}</b> &nbsp; STRONG: <b>{}</b> &nbsp; '
                       'Nonzero summands: {} &nbsp; Positive factor terms: {}</p>'.format(
                           'PASS' if output['weak'] else 'FAIL',
                           'PASS' if output['strong'] else 'FAIL',
                           len(results), output['factor_count']))
        for number, s in enumerate(results, 1):
            section.append('<h4 style="margin:15px 0 7px;color:#143954;">'
                           'Summand {}: (λ′, μ′) = ({}, {}) &nbsp; '
                           '| Multiplicity = {}</h4>'.format(
                               number,part_label(s['lambda_prime']),
                               part_label(s['mu_prime']),s['multiplicity']))
            section.append('<p style="margin:0 0 7px;color:#38556b;">'
                           'β as δ: <b>{}</b> &nbsp; β as γ: <b>{}</b></p>'.format(
                               'YES' if s['has_intersection_as_delta'] else 'no',
                               'YES' if s['has_intersection_as_gamma'] else 'no'))
            section.append('<div style="overflow-x:auto;"><table style="{}">'.format(table))
            section.append('<thead><tr>{}</tr></thead><tbody>'.format(''.join(
                '<th style="{}">{}</th>'.format(head,h) for h in
                ('#','γ','δ','p','q','Nλ','Nμ','Product','β present?'))))
            for j, t in enumerate(s['factors'], 1):
                mark = (t['delta'] == beta or t['gamma'] == beta)
                values = (j,part_label(t['gamma']),part_label(t['delta']),
                          t['p'],t['q'],t['left_coefficient'],
                          t['right_coefficient'],t['contribution'],
                          'YES' if mark else '—')
                section.append('<tr>{}</tr>'.format(''.join(
                    '<td style="{}">{}</td>'.format(cell,html_escape(str(v)))
                    for v in values)))
            section.append('</tbody></table></div>')
            section.append('<p style="margin:7px 0 8px;color:#243e52;">'
                           'Sum of products = <b>{}</b> (matches layer multiplicity).</p>'.format(
                               sum(t['contribution'] for t in s['factors'])))
        section.append('</div>')
        display(HTML(''.join(section)))

    return output

# Alias for callers who prefer the other name.
show_all_top_contributions = show_top_layer_factors


### Interactive investigation of a single outer pair

`show_top_layer` lists all top summands. `inspect_top_pair` reveals individual ordered $(\delta,\gamma,p,q)$ witnesses. `test_intersection_properties` evaluates the weak and strong properties over **all** nonzero top summands, not merely the first one.

**Full visible breakdown:** Run `show_top_layer_factors(LAMBDA, MU)` to show **every** top-layer summand and all positive `(gamma, delta, p, q)` factor combinations, LR coefficients, and multiplicity totals. The displayed table highlights whether the intersection is gamma or delta. Set `render="text"` for console output or `render="none"` for programmatic access.


In [9]:
# Edit these two partitions to inspect all factor values at the maximum layer.
LAMBDA = (2,)
MU = (2, 2)

TOP_K, TOP_SUMMANDS = show_top_layer(LAMBDA, MU)

# NEW: automatically display EVERY positive (gamma, delta, p, q)
# for EVERY top-layer (lambda_prime, mu_prime) summand.
TOP_FACTOR_BREAKDOWN = show_top_layer_factors(LAMBDA, MU, render='html')

# Existing weak/strong checks are retained.
TOP_INTERSECTION_CHECK = test_intersection_properties(LAMBDA, MU)

# Alternative text-only output:
# TOP_FACTOR_BREAKDOWN = show_top_layer_factors(LAMBDA, MU, render='text')


lambda = (2,) mu = (2, 2) intersection = (2,)
Maximum weighted layer k = 5  | nonzero top summands = 2
  (lambda_prime, mu_prime) = ((), (1,)) multiplicity = 2
  (lambda_prime, mu_prime) = ((), ()) multiplicity = 1


#,γ,δ,p,q,Nλ,Nμ,Product,β present?
1,(1),∅,1,2,1,1,1,—
2,(2),∅,0,1,1,1,1,YES
#,γ,δ,p,q,Nλ,Nμ,Product,β present?
1,(1),(1),0,2,1,1,1,—


lambda = (2,) mu = (2, 2) beta = lambda ∩ mu = (2,) | top k = 5
WEAK: PASS | STRONG: FAIL | summands with beta = 1 / 2
  lambda_prime = () mu_prime = (1,) mult = 2 | delta: no | gamma: YES
  lambda_prime = () mu_prime = () mult = 1 | delta: no | gamma: no


### Exhaustive scan and CSV outputs

The scan tests **all ordered pairs** with $|\lambda|\le n$, $|\mu|\le m$, and checks each nonzero summand in their top weighted layer. It prints pass rates and examples of failures and can export two CSV files: one row per outer pair, and one row per top summand.

Set `RUN_EXHAUSTIVE_SCAN=True` and configure the inclusive degree limits. Empty intersections are included by default; run again with the option disabled for a nontrivial-only statistic. Larger limits may be expensive in SageMath.

The scan is disabled by default here because the batch LaTeX exporter below already iterates through every pair. Set `RUN_EXHAUSTIVE_SCAN=True` if you also want the separate weak/strong statistics.


In [10]:
RUN_EXHAUSTIVE_SCAN = False
MAX_LAMBDA_DEGREE = 4
MAX_MU_DEGREE = 4
INCLUDE_EMPTY_INTERSECTION = True

if RUN_EXHAUSTIVE_SCAN:
    INTERSECTION_SCAN = scan_intersection_properties(
        MAX_LAMBDA_DEGREE,
        MAX_MU_DEGREE,
        include_empty_intersection=INCLUDE_EMPTY_INTERSECTION,
        max_examples=10,
        csv_prefix='intersection_tests/theone_degree_le4',
    )


## Batch LaTeX export (all ordered pairs, up to degree 9)

Run this cell to **write the LaTeX document**, not just display a single example.
The default `MAX_DEGREE=9` covers every partition of sizes 0 through 9 in
**both** positions (9,409 ordered pairs). It may take time in SageMath.

The exported file contains, for each pair in turn:

- The two-row extremal table ($k=k_{\max}$ and $k=0$).
- The intersection $\beta$ once.
- Term 1, Term 2, ... with all positive $(\gamma,\delta,p,q)$ realizations.

The dagger legend is printed once in the preamble. The `.tex` file contains
all tables. You can also print the full raw LaTeX code by setting
`PRINT_FULL_LATEX_SOURCE=True`, but doing so for 9,409 pairs produces a very
large notebook output.

In [11]:
# Run this cell in a SageMath kernel to export the complete LaTeX document.
MAX_DEGREE = 9
EXPORT_ALL_DEGREES_TO_LATEX = True
INCLUDE_SWAPPED_PAIRS = True     # True: all ordered pairs (lambda, mu)
PRINT_FULL_LATEX_SOURCE = False  # True: print all LaTeX code into the output (VERY large)
LATEX_OUTPUT_PATH = f"tex_tables/all_pairs_degree_le{MAX_DEGREE}.tex"

if EXPORT_ALL_DEGREES_TO_LATEX:
    ALL_PAIRS_LATEX_FILE = generate_concise_latex_all_pairs(
        max_degree=MAX_DEGREE,
        filename=LATEX_OUTPUT_PATH,
        include_swapped_pairs=INCLUDE_SWAPPED_PAIRS,
        print_latex_source=PRINT_FULL_LATEX_SOURCE,
        preview_chars=800,
        verify=True,
    )
    try:
        from IPython.display import FileLink, display
        display(FileLink(str(ALL_PAIRS_LATEX_FILE)))
    except ImportError:
        pass


LaTeX document written: /Users/pabo/Desktop/IP_Contents/08 SocleProject/SocleCalculator/src/tex_tables/all_pairs_degree_le9.tex
Size: 15,665,657 bytes
LaTeX source preview (full source in the .tex file):
\documentclass[a4paper,11pt]{article}
\usepackage[margin=22mm]{geometry}
\usepackage[T1]{fontenc}
\usepackage{lmodern}
\usepackage{amsmath,amssymb}
\usepackage{array,booktabs,longtable,needspace}
\setlength{\LTleft}{0pt}
\setlength{\LTright}{0pt}
\title{Extremal socle layers and top-layer realizations (degree <= 9)}
\author{}
\date{}
\begin{document}
\maketitle
\noindent Below each table, every row lists one positive top-layer realization $(\gamma,\delta,p,q)$. A $\dagger$ marks $\gamma$ or $\delta$ equal to $\beta=\lambda\cap\mu$.\par\medskip

\noindent Only the initial layer $k=0$ and the highest nonzero layer $k=k_{\max}$ are shown.\\[8pt]
\Needspace{17\baselineskip}
\begingroup
\small
\setlength{\LTpre}{6pt}
\setlength{\LTpost}{6pt}
\begin{longtable}{@{}p{0.14\linewidth}@{\hspace{0

/Users/pabo/Desktop/IP_Contents/08 SocleProject/SocleCalculator/src/tex_tables/all_pairs_degree_le9.tex